# Gradient Boosting (Derived), XGBoost, LightGBM, CatBoost and Tuning

Notebook for **Chapter 7** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 The Idea: Fix the Leftovers (and AdaBoost in One Page)

## 7.2 Gradient Boosting Derived: Gradient Descent on Predictions

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

class MyGB:
    def __init__(s, n_trees=100, lr=0.1, depth=3): s.B, s.lr, s.d = n_trees, lr, depth
    def fit(s, X, y):
        s.f0 = y.mean(); F = np.full(len(y), s.f0); s.trees = []
        for b in range(s.B):
            t = DecisionTreeRegressor(max_depth=s.d, random_state=0).fit(X, y - F)  # fit residuals
            F += s.lr * t.predict(X); s.trees.append(t)
        return s
    def predict(s, X):
        return s.f0 + s.lr * sum(t.predict(X) for t in s.trees)

## 7.3 Learning Rate, Number of Trees, Subsampling and Early Stopping

## 7.4 XGBoost, LightGBM and CatBoost: What Each One Changes

In [ ]:
import lightgbm as lgb
model = lgb.LGBMRegressor(learning_rate=0.05, n_estimators=5000, num_leaves=15,
                          min_child_samples=50, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, reg_lambda=1.0, random_state=0)
# model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(100)])

## 7.5 Tuning Order, Constraints, Custom Losses and Limits

## 7.6 More Worked Boosting Examples: Log Loss, Gain Arithmetic, Early Stopping, Custom Loss

Sections 7.2 and 7.4 gave one hand example each. This section adds four more of different flavours, all run for real. Note on libraries: XGBoost, LightGBM and CatBoost are not installed in the environment where this text was verified (and there was no network), so everything below uses the exact formulas of 7.4 written in NumPy, plus scikit-learn. Any call to the three libraries elsewhere in this chapter is *not run here*; the arithmetic in this section is what those libraries compute for these objectives, but a library cross-check is not run here.

In [ ]:
import numpy as np
x = np.arange(1, 7); y = np.array([0, 0, 1, 0, 1, 1.0])
lam, eta = 1.0, 0.5
F = np.zeros(6)
def logloss(F):
    p = 1 / (1 + np.exp(-F)); return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
print("round 0: log loss %.4f" % logloss(F))
for rnd in (1, 2):
    p = 1 / (1 + np.exp(-F)); g = p - y; h = p * (1 - p)
    print("round", rnd, "p =", np.round(p, 3), "g =", np.round(g, 3))
    best = None
    for k in range(1, 6):                      # split after row k
        GL, HL, GR, HR = g[:k].sum(), h[:k].sum(), g[k:].sum(), h[k:].sum()
        gain = 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - (GL + GR)**2 / (HL + HR + lam))
        if best is None or gain > best[0]: best = (gain, k, -GL / (HL + lam), -GR / (HR + lam))
    gain, k, wl, wr = best
    print("  best split after row %d, gain %.4f, leaf values %.4f and %.4f" % (k, gain, wl, wr))
    F = F + eta * np.where(x <= k, wl, wr)
    print("  new F =", np.round(F, 4))
    print("  log loss %.4f" % logloss(F))

round 0: log loss 0.6931
round 1 p = [0.5 0.5 0.5 0.5 0.5 0.5] g = [ 0.5  0.5 -0.5  0.5 -0.5 -0.5]
  best split after row 2, gain 0.5833, leaf values -0.6667 and 0.5000
  new F = [-0.3333 -0.3333  0.25    0.25    0.25    0.25  ]
  log loss 0.6057
round 2 p = [0.417 0.417 0.562 0.562 0.562 0.562] g = [ 0.417  0.417 -0.438  0.562 -0.438 -0.438]
  best split after row 4, gain 0.4880, leaf values -0.4848 and 0.5868
  new F = [-0.5757 -0.5757  0.0076  0.0076  0.5434  0.5434]
  log loss 0.5324


In [ ]:
import numpy as np
x = np.arange(1, 9); y = np.array([2, 3, 3, 2, 8, 9, 7, 9.0])
F0 = y.mean(); g = F0 - y; h = np.ones(8)       # squared error: g = F - y, h = 1
print("F0 = %.3f  g = %s" % (F0, np.round(g, 3)))
def gain(k, lam, gamma):
    GL, HL, GR, HR = g[:k].sum(), h[:k].sum(), g[k:].sum(), h[k:].sum()
    return 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - (GL + GR)**2 / (HL + HR + lam)) - gamma
print("split after row k:    " + "  ".join("%6d" % k for k in range(1, 8)))
for lam in (0, 1, 10):
    print("gain, lambda=%-2d g=0:  " % lam + "  ".join("%6.3f" % gain(k, lam, 0) for k in range(1, 8)))
lam = 1
k = 4
GL, HL, GR, HR = g[:k].sum(), h[:k].sum(), g[k:].sum(), h[k:].sum()
print("split after row 4: G_L %.3f H_L %d G_R %.3f H_R %d" % (GL, HL, GR, HR))
for lam in (0, 1, 10):
    print("lambda=%-2d leaf values: left %.4f right %.4f" % (lam, -GL / (HL + lam), -GR / (HR + lam)))
def best_inside(lo, hi, lam, gamma):
    # best split inside rows lo..hi-1 (python slice), same g and h as above
    best = (-1e9, None)
    for k in range(lo + 1, hi):
        GL, HL, GR, HR = g[lo:k].sum(), h[lo:k].sum(), g[k:hi].sum(), h[k:hi].sum()
        gn = 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - (GL + GR)**2 / (HL + HR + lam)) - gamma
        best = max(best, (gn, k))
    return best
for lam in (0, 1):
    for gamma in (0, 1, 3):
        bl, br = best_inside(0, 4, lam, gamma), best_inside(4, 8, lam, gamma)
        print("lambda=%d gamma=%d  left child net gain %7.3f (after row %d), right child %7.3f (after row %d)" % (lam, gamma, bl[0], bl[1], br[0], br[1]))

F0 = 5.375  g = [ 3.375  2.375  2.375  3.375 -2.625 -3.625 -1.625 -3.625]
split after row k:         1       2       3       4       5       6       7
gain, lambda=0  g=0:   6.509  11.021  17.604  33.062  21.004   9.188   7.509
gain, lambda=1  g=0:   3.560   7.872  13.753  26.450  16.410   6.562   4.106
gain, lambda=10 g=0:   0.853   2.411   4.740   9.446   5.655   2.010   0.984
split after row 4: G_L 11.500 H_L 4 G_R -11.500 H_R 4
lambda=0  leaf values: left -2.8750 right 2.8750
lambda=1  leaf values: left -2.3000 right 2.3000
lambda=10 leaf values: left -0.8214 right 0.8214
lambda=0 gamma=0  left child net gain   0.167 (after row 3), right child   0.375 (after row 7)
lambda=0 gamma=1  left child net gain  -0.833 (after row 3), right child  -0.625 (after row 7)
lambda=0 gamma=3  left child net gain  -2.833 (after row 3), right child  -2.625 (after row 7)
lambda=1 gamma=0  left child net gain  -2.125 (after row 3), right child  -1.657 (after row 5)
lambda=1 gamma=1  left child net gain

In [ ]:
import numpy as np
from sklearn.datasets import make_friedman1
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as mse
X, y = make_friedman1(n_samples=1500, noise=2.0, random_state=0)
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.5, random_state=0)
Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.5, random_state=0)
m = GradientBoostingRegressor(n_estimators=600, learning_rate=0.1, max_depth=3, random_state=0).fit(Xtr, ytr)
tr = [mse(ytr, p) for p in m.staged_predict(Xtr)]
va = [mse(yva, p) for p in m.staged_predict(Xva)]
te = [mse(yte, p) for p in m.staged_predict(Xte)]
best = int(np.argmin(va)) + 1
print("best number of trees on validation:", best)
print("validation MSE at best %.3f, at 600 %.3f" % (va[best - 1], va[-1]))
print("test MSE at best %.3f, at 600 %.3f" % (te[best - 1], te[-1]))
print("train MSE at best %.3f, at 600 %.3f" % (tr[best - 1], tr[-1]))
# patience rule: stop when validation has not improved for 20 rounds
bestv, since = 1e9, 0
for i, v in enumerate(va, 1):
    if v < bestv - 1e-12: bestv, since, arg = v, 0, i
    else: since += 1
    if since == 20: print("patience 20 would stop at round %d and keep round %d" % (i, arg)); break

best number of trees on validation: 67
validation MSE at best 5.091, at 600 5.675
test MSE at best 5.580, at 600 5.743
train MSE at best 2.710, at 600 0.213
patience 20 would stop at round 87 and keep round 67


In [ ]:
import numpy as np
from sklearn.datasets import make_friedman1
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
X, y = make_friedman1(n_samples=2000, noise=2.0, random_state=1)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=1)

def boost(X, y, k_under, rounds=150, eta=0.1):
    """Newton boosting for l = w (y-F)^2 with w = k_under if y > F else 1. Leaf = -G/H."""
    F = np.full(len(y), y.mean()); f0 = F[0]; trees = []
    for _ in range(rounds):
        w = np.where(y > F, k_under, 1.0)
        g = -2 * w * (y - F); h = 2 * w
        t = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X, -g / h, sample_weight=h)  # leaf = sum(-g)/sum(h)
        F = F + eta * t.predict(X); trees.append(t)
    return lambda Z: f0 + eta * sum(t.predict(Z) for t in trees)

def cost(y, F, k): return np.mean(np.where(y > F, k, 1.0) * (y - F) ** 2)
for k in (1, 3):
    pred = boost(Xtr, ytr, k)(Xte)
    print("trained with weight %d: under-predicted %.1f%% of test rows, mean absolute error %.2f, asymmetric cost (weight 3) %.2f"
          % (k, 100 * np.mean(yte > pred), np.mean(np.abs(yte - pred)), cost(yte, pred, 3)))

trained with weight 1: under-predicted 50.4% of test rows, mean absolute error 1.87, asymmetric cost (weight 3) 11.26
trained with weight 3: under-predicted 39.7% of test rows, mean absolute error 2.01, asymmetric cost (weight 3) 10.29


In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 400); y = 0.5 * x + rng.normal(0, 1.5, 400)
grid = np.linspace(0, 10, 400).reshape(-1, 1)
for cst in (None, [1]):
    m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, monotonic_cst=cst, random_state=0).fit(x.reshape(-1, 1), y)
    p = m.predict(grid)
    print("monotonic_cst =", cst, "downward steps on grid:", int((np.diff(p) < -1e-12).sum()), " total drop %.2f" % abs(np.diff(p)[np.diff(p) < 0].sum()))

monotonic_cst = None downward steps on grid: 60  total drop 33.87
monotonic_cst = [1] downward steps on grid: 0  total drop 0.00


## 7.7 The Bridge to Deep Learning for Tables

Everything in this chapter ends in a plain question: should you put a neural network on a spreadsheet? Published comparisons keep finding that tuned gradient boosting is as good or better than neural networks on most medium-size tabular data, and much cheaper to train. Networks win with huge data, text or image columns mixed in, multi-task output, or when you need embeddings shared with other models. This section shows *why* trees have the edge with a small experiment you can reproduce (scikit-learn only; PyTorch and the specialised tabular networks TabNet, FT-Transformer and TabPFN are not installed here, so they are *not run here*).

In [ ]:
import numpy as np, warnings; warnings.filterwarnings("ignore")
from sklearn.ensemble import HistGradientBoostingClassifier as H
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0); n = 6000
X = np.c_[rng.lognormal(0, 1.5, (n, 3)), rng.normal(size=(n, 3)), rng.normal(size=(n, 14))]   # 3 skewed, 3 normal, 14 noise
z = (X[:, 0] > 1.5) * 1.5 + np.sin(3 * X[:, 3]) + (X[:, 1] > 1) * (X[:, 4] > 0) * 1.5 + 0.3 * X[:, 5]
y = (z + rng.normal(0, 0.5, n) > 1.2).astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, random_state=0)
print("positive rate", y.mean().round(3))
mods = {"HistGB": H(random_state=0),
        "MLP raw": MLPClassifier((64, 64), max_iter=300, random_state=0),
        "MLP standardised": make_pipeline(StandardScaler(), MLPClassifier((64, 64), max_iter=300, random_state=0)),
        "MLP quantile": make_pipeline(QuantileTransformer(output_distribution="normal", random_state=0), MLPClassifier((64, 64), max_iter=300, random_state=0))}
for k, m in mods.items(): print(f"{k:18s} test accuracy {m.fit(Xtr, ytr).score(Xte, yte):.3f}")

positive rate 0.422
HistGB             test accuracy 0.884
MLP raw            test accuracy 0.695
MLP standardised   test accuracy 0.648
MLP quantile       test accuracy 0.693


## Chapter recap

## Practice set

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch07.html